# Final PCR Prediction (Classification)

In [3]:
import pandas as pd
import joblib
import numpy as np

### Preprocessing

#### Before Preprocessing

In [4]:
dataset_x_all = pd.read_csv("../datasets/FinalTestDataset2025.csv")

#### Type-specific preprocessing

In [5]:
dataset_x_all = dataset_x_all.drop("ID",axis=1)
dataset_x_all['original_shape_VoxelVolume'] = dataset_x_all['original_shape_VoxelVolume'].astype(np.float64)
categorical_names = dataset_x_all.select_dtypes(np.int64).columns

continuous_names = dataset_x_all.select_dtypes(np.float64).columns

dataset_x_all = dataset_x_all.replace([999], np.nan)

#### General Preprocessing

In [6]:
Xs = dataset_x_all.copy()

lower_quartile = Xs[continuous_names].quantile(0.25)
upper_quartile = Xs[continuous_names].quantile(0.75)
iqr = upper_quartile - lower_quartile

lower_bound = lower_quartile - 1.5 * iqr
upper_bound = upper_quartile + 1.5 * iqr

Xs[continuous_names] = np.where(Xs[continuous_names] < lower_bound, lower_bound, Xs[continuous_names])
Xs[continuous_names] = np.where(Xs[continuous_names] > upper_bound, upper_bound, Xs[continuous_names])

In [7]:
categorical_ranges = {
    "pCR (outcome)": (0,1),
    "ER": (0, 1),
    "PgR": (0, 1),
    "HER2": (0, 1),
    "TrippleNegative": (0, 1),
    "ChemoGrade": (1, 3),
    "Proliferation": (1, 3),
    "HistologyType": (1, 2),
    "LNStatus": (0, 1),
    "TumourStage": (1, 4),
    "Gene": (0, 1)
}

In [8]:
data_train_cat = pd.concat([Xs[categorical_names]], axis=1)
for col in data_train_cat.columns:
    
    # Truncate data to be within range for categorical features
    data_train_cat[col] = np.where(data_train_cat[col] > categorical_ranges[col][1], categorical_ranges[col][1],  data_train_cat[col])
    data_train_cat[col] = np.where(data_train_cat[col] < categorical_ranges[col][0], categorical_ranges[col][0],  data_train_cat[col])

In [9]:
imputer_knn = joblib.load('../exports/pcr_imputer_knn.pkl')
z_scaler = joblib.load('../exports/pcr_z_scaler_knn.pkl')

Xs = pd.DataFrame(imputer_knn.transform(Xs), columns=Xs.columns, index=Xs.index)
Xs[continuous_names] = z_scaler.transform(Xs[continuous_names])

Xs

,Age,ER,PgR,HER2,TrippleNegative,ChemoGrade,Proliferation,HistologyType,LNStatus,TumourStage,...,original_glszm_SmallAreaHighGrayLevelEmphasis,original_glszm_SmallAreaLowGrayLevelEmphasis,original_glszm_ZoneEntropy,original_glszm_ZonePercentage,original_glszm_ZoneVariance,original_ngtdm_Busyness,original_ngtdm_Coarseness,original_ngtdm_Complexity,original_ngtdm_Contrast,original_ngtdm_Strength
0,-0.467876,1.0,1.0,0.0,0.0,3.0,2.0,1.0,1.0,2.0,...,0.702989,0.745200,0.594476,0.068281,0.260496,0.286070,-0.752533,-0.595326,-0.795654,-0.755758
1,-1.052151,1.0,1.0,0.0,0.0,3.0,2.0,1.0,0.0,2.0,...,1.536606,1.605549,-0.982034,1.408426,-0.652034,-0.798138,0.148099,-0.998680,-0.968302,0.295097
2,0.116399,0.0,0.0,0.0,1.0,2.0,1.0,1.0,1.0,2.0,...,0.125865,0.149568,1.025033,0.912002,-0.503546,0.501557,-0.770612,0.268746,-0.155536,-0.766570
3,-0.565256,1.0,1.0,0.0,0.0,2.0,1.0,1.0,1.0,3.0,...,-0.103651,-0.087308,1.158036,1.654196,-0.632070,-0.181270,-0.636294,-0.225860,-0.581991,-0.617244
4,-1.246910,0.0,0.0,1.0,0.0,2.0,2.0,1.0,1.0,2.0,...,0.880540,0.928445,-0.800703,1.998922,-0.796359,-0.918312,1.337029,-0.912650,-0.944994,1.674213
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
128,1.022025,0.0,0.0,1.0,0.0,2.0,1.0,1.0,0.0,3.0,...,1.050797,1.104149,-0.150775,0.470189,-0.649916,-0.231308,-0.613478,-0.469373,-0.664040,-0.551811
129,-0.555518,0.0,0.0,0.0,1.0,3.0,3.0,1.0,1.0,4.0,...,0.307912,0.337453,0.465571,-0.846942,1.413908,-0.328667,-0.578082,-0.876846,-0.887411,-0.512924
130,-0.730800,0.0,0.0,0.0,1.0,3.0,2.0,1.0,0.0,2.0,...,0.383118,0.415071,0.123230,0.210018,-0.252997,0.237441,-0.734315,-0.226830,-0.452402,-0.717604
131,-1.431930,0.0,0.0,0.0,1.0,3.0,3.0,1.0,1.0,3.0,...,-0.059788,-0.042038,1.374856,0.177283,-0.312832,1.164041,-0.819199,0.646037,0.367310,-0.818875


Notes: HistologyType 1.5????????????????????????????????????????????????????????????????????????? LMAO

### Feature Selection

### Prediction

In [ ]:
models = joblib.load('../exports/pcr_models.pkl')

for key in models.keys():
    print(f"{key}: {models[key].predict(Xs)}")

# Then choose a model ig
# And write to csv don't forget

Logistic Regression: [0. 1. 0. 0. 1. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 1. 1. 1. 1.
 1. 0. 1. 1. 0. 0. 0. 1. 0. 1. 0. 0. 1. 0. 1. 0. 0. 0. 0. 0. 1. 0. 1. 0.
 0. 0. 0. 1. 0. 1. 1. 1. 1. 0. 0. 1. 0. 1. 0. 1. 1. 0. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 0. 0. 1. 1. 1. 1. 1. 1. 0. 1. 0. 0. 0. 0. 0. 0. 0.
 0. 1. 1. 0. 1. 0. 0. 1. 0. 1. 0. 0. 0. 1. 0. 1. 1. 1. 1. 0. 1. 1. 1. 0.
 1. 0. 1. 0. 0. 1. 1. 0. 1. 0. 0. 0. 1.]
Naive Bayes (Gaussian): [0. 0. 1. 1. 0. 0. 0. 0. 0. 0. 1. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 1. 1. 0. 0. 1. 1. 0. 1. 1. 1. 0. 1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.
 0. 0. 1. 1. 1. 1. 0. 1. 1. 0. 0. 1. 0. 1. 0. 1. 1. 0. 1. 1. 0. 1. 1. 0.
 1. 1. 0. 1. 1. 1. 1. 0. 0. 1. 1. 1. 0. 1. 1. 0. 0. 0. 1. 0. 1. 0. 0. 1.
 0. 0. 1. 0. 0. 1. 1. 1. 1. 1. 0. 0. 0. 1. 0. 1. 1. 0. 1. 0. 1. 1. 0. 0.
 1. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 1. 1.]
KNN (k=3): [0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 1. 1. 0. 1. 1. 0. 0. 0. 1. 0. 0. 1. 0. 0.
 1. 1. 0. 0. 1. 1. 0. 1. 1. 1. 0. 0. 1. 0. 0. 0. 0. 1. 0. 1